# Classification metrics and cross-validation

Accuracy can hide mistakes in a minority class. Compare a majority-class baseline with logistic regression, use stratified cross-validation on training data, and inspect held-out precision/recall and ranking metrics.

The dataset is generated for teaching. It needs no download or GPU. Use the base requirements and run from a fresh kernel.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, PrecisionRecallDisplay,
    accuracy_score, balanced_accuracy_score, average_precision_score,
    classification_report, roc_auc_score)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

SEED = 42
X, y = make_classification(n_samples=600, n_features=10, n_informative=5,
    n_redundant=2, weights=[0.85, 0.15], class_sep=1.0, random_state=SEED)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED)
print('Training class counts:', np.bincount(y_train))


## 1. Compare training-fold results

Each scaler is fitted inside its fold. Use the same folds for both estimators. Balanced accuracy averages recall over classes; average precision summarizes the precision/recall ranking of class `1`. These address different questions.


In [ ]:
models = {
    'majority baseline': DummyClassifier(strategy='most_frequent'),
    'logistic regression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=500)),
}
cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=SEED)
scoring = {'accuracy': 'accuracy', 'balanced_accuracy': 'balanced_accuracy',
           'average_precision': 'average_precision', 'roc_auc': 'roc_auc'}
rows = []
for name, estimator in models.items():
    scores = cross_validate(estimator, X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    row = {'model': name}
    row.update({metric: scores['test_' + metric].mean() for metric in scoring})
    rows.append(row)
display(pd.DataFrame(rows).set_index('model').round(3))


## 2. Evaluate the chosen learning model once on held-out data

The decision threshold is fixed at `0.5` here. Threshold selection, if needed, belongs in training/validation data, not in repeated inspection of this test set.


In [ ]:
model = models['logistic regression'].fit(X_train, y_train)
probability = model.predict_proba(X_test)[:, 1]
prediction = (probability >= 0.5).astype(int)
print(classification_report(y_test, prediction, zero_division=0))
print({'accuracy': round(accuracy_score(y_test, prediction), 3),
       'balanced_accuracy': round(balanced_accuracy_score(y_test, prediction), 3),
       'average_precision': round(average_precision_score(y_test, probability), 3),
       'roc_auc': round(roc_auc_score(y_test, probability), 3)})


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
ConfusionMatrixDisplay.from_predictions(y_test, prediction, ax=axes[0], colorbar=False)
PrecisionRecallDisplay.from_predictions(y_test, probability, ax=axes[1])
axes[1].axhline(y_test.mean(), linestyle='--', color='gray', label='Positive-class prevalence')
axes[1].legend()
fig.tight_layout()
plt.show()


## Practice

Change the generated class proportions and rerun the complete experiment. Explain why the majority baseline's accuracy can improve while minority-class recall remains zero. For time-dependent or grouped observations, replace random folds with a split that respects those relationships.

**Reference:** [scikit-learn: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).
